# PaceAlgo ML — Notebook 02: Feature Engineering

**Zweck:** Aus den OHLCV-Parquets (aus Notebook 01) eine Feature-Matrix pro Symbol/Timeframe berechnen.

**Pro Bar werden ~30 Features berechnet:**
- **Trend (7):** EMA distances, slopes, alignment, ADX
- **Momentum (5):** RSI, StochRSI, ROC, MACD-hist, momentum composite
- **Volatility (5):** ATR%, ATR percentile, BB width, vol compression, realized vol
- **Structure (4):** swing distances, BOS bullish/bearish
- **Volume (2):** relative volume, volume z-score
- **Session (2):** hour sin/cos (cyclical encoding)
- **HTF Context (6):** 1H + 4H alignment, RSI, ATR percentile
- **Macro (6):** VIX, DXY, TNX levels + 5d changes

Alle Features sind **ATR-normalisiert oder dimensionslos** — das Modell lernt universelle Marktstruktur, nicht asset-spezifische Preisniveaus.

**Output:** `data_cache/processed/<symbol>_<tf>_features.parquet`

**Laufzeit:** ~5-8 Min für alle Symbole.


## 1. Environment Setup

In [ ]:
import sys, os
IS_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IS_COLAB}')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    os.chdir(PROJECT_ROOT)
    # Always pull latest code
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)

sys.path.insert(0, PROJECT_ROOT)
# Clear cached imports
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from core.config import (
    DATA_RAW, DATA_PROCESSED,
    CRYPTO_SYMBOLS, FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES, HTF_CONTEXT_TIMEFRAMES,
)
from core.features import compute_features, attach_macro, attach_htf_context

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ALL_SYMBOLS = CRYPTO_SYMBOLS + FX_SYMBOLS + METAL_SYMBOLS
print(f'Will process {len(ALL_SYMBOLS)} symbols × {len(PRIMARY_TIMEFRAMES)} primary TFs')
print(f'Symbols: {ALL_SYMBOLS}')
print(f'Primary TFs: {PRIMARY_TIMEFRAMES}')
print(f'HTF context: {HTF_CONTEXT_TIMEFRAMES}')
print(f'Output: {DATA_PROCESSED}')

## 2. Load Macro Daily (shared across symbols)

In [ ]:
macro_path = DATA_RAW / 'macro_daily.parquet'
if macro_path.exists():
    macro = pd.read_parquet(macro_path)
    print(f'Macro loaded: {len(macro):,} daily obs, columns: {list(macro.columns)}')
    display(macro.tail(3))
else:
    macro = pd.DataFrame()
    print('No macro daily found — features will be computed without macro')

## 3. Compute Features (primary + HTF context + macro)

Pro Symbol:
1. Berechne Features auf 1H und 4H (HTF context)
2. Für jedes primary TF (5M, 15M): berechne Features, hänge HTF context an, hänge Macro an
3. Speichere in `data_cache/processed/<symbol>_<tf>_features.parquet`

In [ ]:
def load_ohlcv(symbol: str, tf: str):
    p = DATA_RAW / f'{symbol}_{tf}.parquet'
    if not p.exists():
        return None
    return pd.read_parquet(p)

summary_rows = []
for symbol in tqdm(ALL_SYMBOLS, desc='Symbols'):
    # ── Step 1: HTF context features (1H + 4H) ──
    htf_features = {}
    for htf_tf in HTF_CONTEXT_TIMEFRAMES:
        df_htf = load_ohlcv(symbol, htf_tf)
        if df_htf is None or df_htf.empty:
            print(f'  WARN {symbol} {htf_tf}: missing — skip HTF for this symbol')
            htf_features[htf_tf] = pd.DataFrame()
        else:
            htf_features[htf_tf] = compute_features(df_htf)

    # ── Step 2: Primary TFs (5M + 15M) ──
    for tf in PRIMARY_TIMEFRAMES:
        out_path = DATA_PROCESSED / f'{symbol}_{tf}_features.parquet'
        if out_path.exists():
            print(f'  skip {symbol} {tf} (cached)')
            continue

        df = load_ohlcv(symbol, tf)
        if df is None or df.empty:
            print(f'  SKIP {symbol} {tf}: no OHLCV')
            continue

        # Compute primary features
        feats = compute_features(df)

        # Attach HTF context (1H + 4H)
        feats = attach_htf_context(
            feats,
            htf_features.get('1h', pd.DataFrame()),
            htf_features.get('4h', pd.DataFrame()),
        )

        # Attach macro
        feats = attach_macro(feats, macro)

        # Add identifier columns (useful when stacking across symbols later)
        feats['symbol'] = symbol
        feats['timeframe'] = tf

        # Save
        feats.to_parquet(out_path, compression='zstd')
        valid_rows = feats.dropna(subset=[c for c in feats.columns if c not in ('symbol','timeframe')]).shape[0]
        summary_rows.append({
            'symbol': symbol, 'tf': tf,
            'total_rows': len(feats),
            'valid_rows': valid_rows,
            'num_features': len([c for c in feats.columns if c not in ('symbol','timeframe')]),
            'size_MB': out_path.stat().st_size / 1024**2,
        })
        print(f'  OK  {symbol} {tf}: {valid_rows:,}/{len(feats):,} valid rows × {summary_rows[-1]["num_features"]} features')

print('\nDone.')

## 4. Summary Table

In [ ]:
if summary_rows:
    sdf = pd.DataFrame(summary_rows)
    display(sdf)
    print(f'\nTotal processed files: {len(sdf)}')
    print(f'Total size: {sdf["size_MB"].sum():.1f} MB')
    print(f'Total valid rows across all symbols/TFs: {sdf["valid_rows"].sum():,}')
else:
    print('Nothing processed (already cached or no input data).')

## 5. Sanity Check — BTC 5M Sample

In [ ]:
sample_path = DATA_PROCESSED / 'BTCUSDT_5m_features.parquet'
if sample_path.exists():
    sample = pd.read_parquet(sample_path)
    feature_cols = [c for c in sample.columns if c not in ('symbol', 'timeframe')]
    print(f'Shape: {sample.shape[0]:,} rows × {len(feature_cols)} features')
    print(f'Date range: {sample.index[0]} → {sample.index[-1]}')

    print(f'\nFeature list ({len(feature_cols)}):')
    for i, c in enumerate(feature_cols):
        print(f'  {i+1:2d}. {c}')

    print(f'\nDescriptive stats (last 100k bars):')
    display(sample[feature_cols].tail(100_000).describe().T[['mean','std','min','max']].round(3))

    print(f'\nNaN count per feature (full sample):')
    nan_counts = sample[feature_cols].isna().sum().sort_values(ascending=False)
    display(nan_counts[nan_counts > 0])
else:
    print('BTC 5M features not found — check that notebook 01 ran successfully.')

---

## Nächster Schritt

→ `03_asset_clustering.ipynb`

K-Means Clustering der Symbole auf Volatilitäts-/Trend-Eigenschaften → datengetriebene Asset-Klassen statt Ticker-Hardcoding.